In [1]:
import pandas as pd
import numpy as np
import random
from collections import defaultdict
from tqdm import tqdm

In [2]:
captive_plants_w_oil= pd.read_excel(r'./data/captive_power_plants/Check_point_1.xlsx')
captive_plants = pd.read_excel(r'./data/captive_power_plants/Check_point_7.xlsx')
stranded_emission = captive_plants_w_oil['stranded emission'].sum()

In [3]:
stranded_emission

46767006506.08786

### 0.1 Aggregate plant units

In [4]:
df = captive_plants.copy()

# ========= 1. Define columns =========
feas_cols = [
    'feasibi_solar','feasibi_wind','feasibi_grid','feasibi_ccs','feasibi_bio',
    'feasibi_coaltogas','feasibi_bio_ccs','feasibi_solar_grid', 'feasibi_wind_grid'
]
miti_cols = [
    c for c in df.columns 
    if c.startswith("Miti_") and "biomass" not in c
]
cost_cols = [
    c for c in df.columns 
    if c.startswith("cost_") and "biomass" not in c
]

# ========= 2. Define cost and miti column mapping =========
cost_to_miti = {c: "Miti_" + c.replace("cost_", "") for c in cost_cols}

# ========= 3. Group =========
grouped = df.groupby("Plant_ID_High", dropna=False)

# ========= 4. miti → sum =========
miti_agg = grouped[miti_cols].sum()

# ========= 5. feas → max =========
feas_agg = grouped[feas_cols].max()

# ========= 6. cost → weighted average =========
cost_agg_list = []
for cost_col, miti_col in cost_to_miti.items():
    
    def weighted_avg(g):
        miti = g[miti_col]
        cost = g[cost_col]
        if miti.sum() == 0:
            return np.nan
        return (cost * miti).sum() / miti.sum()
    
    cost_series = grouped.apply(weighted_avg)
    cost_series.name = cost_col
    cost_agg_list.append(cost_series)

cost_agg = pd.concat(cost_agg_list, axis=1)

# ========= 7. Merge =========
df_agg = pd.concat([miti_agg, feas_agg, cost_agg], axis=1).reset_index()

# ========= 8. Build mapping =========
mapping_dict = (
    df.groupby("Plant_ID_High")["Plant_ID"].apply(list).to_dict()
)

# ========= 9. Done =========
print("Number of plants after aggregation:", len(df_agg))
print("Example mapping:", list(mapping_dict.items())[:3])

C:\Users\18470\AppData\Local\Temp\ipykernel_27340\1402885291.py:40: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cost_series = grouped.apply(weighted_avg)


Number of plants after aggregation: 5123
Example mapping: [(1, [13424]), (2, [6073]), (3, [11036, 11037])]


In [5]:
# This chunk is not important; used for data checking.
group_cols = ["plant", "longitude", "latitude", "fuel_class"]

feas_cols = [
    'feasibi_solar','feasibi_wind','feasibi_grid','feasibi_ccs','feasibi_bio',
    'feasibi_coaltogas','feasibi_bio_ccs','feasibi_solar_grid', 'feasibi_wind_grid'
]

# Count unique values of each feas column within each group
check = captive_plants.groupby(group_cols)[feas_cols].nunique()

# Find conflicts (>1)
conflict = check[check > 1]

# Keep only groups with at least one conflicting column
conflict_groups = conflict.dropna(how="all")

print("Number of groups with inconsistent feasibility:", len(conflict_groups))

Number of groups with inconsistent feasibility: 247


### 1. Data preparation

In [6]:
feas_map = {
    'solar': 'feasibi_solar',         # Solar maps to renewable energy column
    'wind':  'feasibi_wind',         # Wind also uses the renewable energy column
    'grid':  'feasibi_grid',
    'ccs':   'feasibi_ccs',
    'bio':   'feasibi_bio',
    'coaltogas': 'feasibi_coaltogas',
    'bio_ccs':  'feasibi_bio_ccs',
    '30solar_grid': 'feasibi_solar_grid',
    '30wind_grid':  'feasibi_wind_grid',
    '40solar_grid': 'feasibi_solar_grid',
    '40wind_grid':  'feasibi_wind_grid',
    '50solar_grid': 'feasibi_solar_grid',
    '50wind_grid':  'feasibi_wind_grid',
    '60solar_grid': 'feasibi_solar_grid',
    '60wind_grid':  'feasibi_wind_grid',
    '70solar_grid': 'feasibi_solar_grid',
    '70wind_grid':  'feasibi_wind_grid',
    '80solar_grid': 'feasibi_solar_grid',
    '80wind_grid':  'feasibi_wind_grid'
}

In [7]:
PLANT_ID_COL = "Plant_ID_High" 
cost_cols = [c for c in df_agg.columns if c.startswith("cost_")]
methods = [c.replace("cost_", "") for c in cost_cols]
#methods = [m for m in methods if m != 'biomass']  # remove biomass, keep bio

records = []
for _, row in df_agg.iterrows():
    pid = row[PLANT_ID_COL]
    for m in methods:
        feas_col = feas_map[m]
        feasible = row[feas_col]
        miti = row[f"Miti_{m}"]              # directly take mitigation value
        if feasible != 1 or miti <= 0:
            continue
        cost_per_t = row[f"cost_{m}"]
        total_cost = cost_per_t * miti
        records.append((pid, m, cost_per_t, miti, total_cost))

df_long = pd.DataFrame(records, columns=[PLANT_ID_COL, "method", "cost_per_t", "miti", "total_cost"])
print("Number of feasible options (long table):", len(df_long))

Number of feasible options (long table):  22285


In [8]:
def pareto_filter(group):
    """
    Apply Pareto filter to a single plant's options:
    retain only non-dominated methods
    """
    plant_id = group.name          # This is the groupby key

    g = group.copy()
    dominated = []

    for i, row_i in g.iterrows():
        for j, row_j in g.iterrows():
            if i == j:
                continue

            # j dominates i
            if (
                (row_j["cost_per_t"] <= row_i["cost_per_t"]) and
                (row_j["miti"] >= row_i["miti"]) and
                (
                    (row_j["cost_per_t"] < row_i["cost_per_t"]) or
                    (row_j["miti"] > row_i["miti"])
                )
            ):
                dominated.append(i)
                break

    out = g.drop(index=dominated)

    # Explicitly add PLANT_ID_COL back (required for pandas 2.x)
    out[PLANT_ID_COL] = plant_id

    return out

In [9]:
df_filtered = (
    df_long
    .groupby(PLANT_ID_COL, group_keys=False)
    .apply(pareto_filter, include_groups=False)
    .reset_index(drop=True)
)
# Remove |cost_per_t| > 500
df_filtered = df_filtered[
    df_filtered["cost_per_t"].abs() <= 500
]

### Linear optimization

In [22]:
import pulp

In [23]:
emission_reduction_rate = 0.2
M_target = stranded_emission * emission_reduction_rate
df_opt = df_filtered.copy()

# ========= (Safety check) =========
required_cols = ["Plant_ID_High", "miti", "total_cost"]
assert all(c in df_opt.columns for c in required_cols), "Missing required columns"

assert (df_opt["miti"] > 0).all(), "Non-positive miti found"
#assert (df_opt["total_cost"] >= 0).all(), "Negative cost found"

assert (df_opt.groupby("Plant_ID_High").size() >= 1).all()

max_miti_possible = df_opt.groupby("Plant_ID_High")["miti"].max().sum()
assert M_target <= max_miti_possible, "M_target exceeds max achievable mitigation"
# =================================


# ============== Modeling =====================
prob = pulp.LpProblem("MinCostMitigation", pulp.LpMinimize)

x = pulp.LpVariable.dicts(
    "x",
    df_opt.index,
    lowBound=0,
    upBound=1,
    cat="Binary"
)

# Objective function
prob += pulp.lpSum(df_opt.loc[i, "total_cost"] * x[i] for i in df_opt.index)

# Each plant can select at most one method (allowed to take no action)
for plant, g in df_opt.groupby("Plant_ID_High"):
    prob += pulp.lpSum(x[i] for i in g.index) <= 1

# Total mitigation constraint
prob += pulp.lpSum(df_opt.loc[i, "miti"] * x[i] for i in df_opt.index) >= M_target

# Solve
prob.solve(
    pulp.PULP_CBC_CMD(
        msg=True,
        timeLimit=600,
        gapRel=0.001
    )
)

# Selected options
solution = df_opt.loc[[i for i in df_opt.index if x[i].value() == 1]].copy()
# All plants
all_plants = pd.DataFrame({"Plant_ID_High": df_agg["Plant_ID_High"].unique()})
# merge
result = all_plants.merge(solution, on="Plant_ID_High", how="left")
# Mark unselected plants as no_action
result["method"] = result["method"].fillna("no_action")

rate_pct = int(emission_reduction_rate * 100)
out_path = f"./data/captive_power_plants/diverse_miti_target/{rate_pct}pct_reduction.xlsx"
result.to_excel(out_path, index=False)

print("Status:", pulp.LpStatus[prob.status])
print("Total cost:", pulp.value(prob.objective))
print("Selected plants:", solution["Plant_ID_High"].nunique())
print("Selected options:", len(solution))
print("Total mitigation achieved:", solution["miti"].sum())
print("Target mitigation:", M_target)
avg_cost_per_mit = solution["total_cost"].sum() / solution["miti"].sum()
print("Average cost per unit mitigation:", avg_cost_per_mit)

Status: Optimal
Total cost: 370095600546.44275
Selected plants: 2372
Selected options: 2372
Total mitigation achieved: 9353401303.4612
Target mitigation: 9353401301.217573
Average cost per unit mitigation: 39.56802328255602


In [24]:
emission_reduction_rate = 0.4
M_target = stranded_emission * emission_reduction_rate
df_opt = df_filtered.copy()

# ========= (Safety check) =========
required_cols = ["Plant_ID_High", "miti", "total_cost"]
assert all(c in df_opt.columns for c in required_cols), "Missing required columns"

assert (df_opt["miti"] > 0).all(), "Non-positive miti found"
#assert (df_opt["total_cost"] >= 0).all(), "Negative cost found"

assert (df_opt.groupby("Plant_ID_High").size() >= 1).all()

max_miti_possible = df_opt.groupby("Plant_ID_High")["miti"].max().sum()
assert M_target <= max_miti_possible, "M_target exceeds max achievable mitigation"
# =================================


# ============== Modeling =====================
prob = pulp.LpProblem("MinCostMitigation", pulp.LpMinimize)

x = pulp.LpVariable.dicts(
    "x",
    df_opt.index,
    lowBound=0,
    upBound=1,
    cat="Binary"
)

# Objective function
prob += pulp.lpSum(df_opt.loc[i, "total_cost"] * x[i] for i in df_opt.index)

# Each plant can select at most one method (allowed to take no action)
for plant, g in df_opt.groupby("Plant_ID_High"):
    prob += pulp.lpSum(x[i] for i in g.index) <= 1

# Total mitigation constraint
prob += pulp.lpSum(df_opt.loc[i, "miti"] * x[i] for i in df_opt.index) >= M_target

# Solve
prob.solve(
    pulp.PULP_CBC_CMD(
        msg=True,
        timeLimit=600,
        gapRel=0.001
    )
)

# Selected options
solution = df_opt.loc[[i for i in df_opt.index if x[i].value() == 1]].copy()
# All plants
all_plants = pd.DataFrame({"Plant_ID_High": df_agg["Plant_ID_High"].unique()})
# merge
result = all_plants.merge(solution, on="Plant_ID_High", how="left")
# Mark unselected plants as no_action
result["method"] = result["method"].fillna("no_action")

rate_pct = int(emission_reduction_rate * 100)
out_path = f"./data/captive_power_plants/diverse_miti_target/{rate_pct}pct_reduction.xlsx"
result.to_excel(out_path, index=False)

print("Status:", pulp.LpStatus[prob.status])
print("Total cost:", pulp.value(prob.objective))
print("Selected plants:", solution["Plant_ID_High"].nunique())
print("Selected options:", len(solution))
print("Total mitigation achieved:", solution["miti"].sum())
print("Target mitigation:", M_target)
avg_cost_per_mit = solution["total_cost"].sum() / solution["miti"].sum()
print("Average cost per unit mitigation:", avg_cost_per_mit)

Status: Optimal
Total cost: 1277188993372.6858
Selected plants: 2421
Selected options: 2421
Total mitigation achieved: 18706802831.846184
Target mitigation: 18706802602.435146
Average cost per unit mitigation: 68.27403938841002


In [25]:
emission_reduction_rate = 0.6
M_target = stranded_emission * emission_reduction_rate
df_opt = df_filtered.copy()

# ========= (Safety check) =========
required_cols = ["Plant_ID_High", "miti", "total_cost"]
assert all(c in df_opt.columns for c in required_cols), "Missing required columns"

assert (df_opt["miti"] > 0).all(), "Non-positive miti found"
#assert (df_opt["total_cost"] >= 0).all(), "Negative cost found"

assert (df_opt.groupby("Plant_ID_High").size() >= 1).all()

max_miti_possible = df_opt.groupby("Plant_ID_High")["miti"].max().sum()
assert M_target <= max_miti_possible, "M_target exceeds max achievable mitigation"
# =================================


# ============== Modeling =====================
prob = pulp.LpProblem("MinCostMitigation", pulp.LpMinimize)

x = pulp.LpVariable.dicts(
    "x",
    df_opt.index,
    lowBound=0,
    upBound=1,
    cat="Binary"
)

# Objective function
prob += pulp.lpSum(df_opt.loc[i, "total_cost"] * x[i] for i in df_opt.index)

# Each plant can select at most one method (allowed to take no action)
for plant, g in df_opt.groupby("Plant_ID_High"):
    prob += pulp.lpSum(x[i] for i in g.index) <= 1

# Total mitigation constraint
prob += pulp.lpSum(df_opt.loc[i, "miti"] * x[i] for i in df_opt.index) >= M_target

# Solve
prob.solve(
    pulp.PULP_CBC_CMD(
        msg=True,
        timeLimit=600,
        gapRel=0.001
    )
)

# Selected options
solution = df_opt.loc[[i for i in df_opt.index if x[i].value() == 1]].copy()
# All plants
all_plants = pd.DataFrame({"Plant_ID_High": df_agg["Plant_ID_High"].unique()})
# merge
result = all_plants.merge(solution, on="Plant_ID_High", how="left")
# Mark unselected plants as no_action
result["method"] = result["method"].fillna("no_action")

rate_pct = int(emission_reduction_rate * 100)
out_path = f"./data/captive_power_plants/diverse_miti_target/{rate_pct}pct_reduction.xlsx"
result.to_excel(out_path, index=False)

print("Status:", pulp.LpStatus[prob.status])
print("Total cost:", pulp.value(prob.objective))
print("Selected plants:", solution["Plant_ID_High"].nunique())
print("Selected options:", len(solution))
print("Total mitigation achieved:", solution["miti"].sum())
print("Target mitigation:", M_target)
avg_cost_per_mit = solution["total_cost"].sum() / solution["miti"].sum()
print("Average cost per unit mitigation:", avg_cost_per_mit)

Status: Optimal
Total cost: 2383565218978.4062
Selected plants: 2569
Selected options: 2569
Total mitigation achieved: 28060204111.61211
Target mitigation: 28060203903.652714
Average cost per unit mitigation: 84.94468570141372


### Least cost

In [33]:
least_cost_solution = (
    df_filtered.loc[df_filtered.groupby("Plant_ID_High")["cost_per_t"].idxmin()]
    .reset_index(drop=True)
)

In [27]:
least_cost_solution.to_excel(r'./data/captive_power_plants/diverse_miti_target/least_cost.xlsx')

In [34]:
b = least_cost_solution['miti'].sum()
a = b/stranded_emission
a

0.19377067917009747

In [35]:
b

9062074613.437012

### Most mitigation

In [29]:
most_miti_solution = (
    df_filtered.loc[df_filtered.groupby("Plant_ID_High")["miti"].idxmax()]
    .reset_index(drop=True)
)

In [30]:
most_miti_solution.to_excel(r'./data/captive_power_plants/diverse_miti_target/most_miti.xlsx')

In [31]:
b = most_miti_solution['miti'].sum()
a = b/stranded_emission
a

0.6961900076677114

### Output result

In [32]:
import pandas as pd
import os

# ========= 1. Input files =========
file_dict = {
    "20pct": "20pct_reduction.xlsx",
    "40pct": "40pct_reduction.xlsx",
    "60pct": "60pct_reduction.xlsx",
    "least_cost": "least_cost.xlsx",
    "most_miti": "most_miti.xlsx"
}

base_path = "./data/captive_power_plants/diverse_miti_target/"

mapping = captive_plants[["Plant_ID_High", "Plant_ID"]]

# ========= 2. Loop =========
for name, file in file_dict.items():
    
    print(f"Processing: {name}")
    
    # Read
    result_tmp = pd.read_excel(os.path.join(base_path, file))
    
    # ======== Step 1: merge method =========
    assign_tmp = mapping.merge(
        result_tmp[["Plant_ID_High", "method"]],
        on="Plant_ID_High",
        how="left"
    )
    
    # Key: avoid NaN
    assign_tmp["method"] = assign_tmp["method"].fillna("no_action")
    
    # ======== Step 2: merge with original data =========
    assign_tmp = assign_tmp.merge(
        captive_plants.drop(columns=["Plant_ID_High"]),
        on="Plant_ID",
        how="left"
    )
    
    # ======== Step 3: extract values =========
    def extract_values(row):
        method = row["method"]
        
        if method == "no_action" or pd.isna(method):
            return pd.Series({
                "cost_per_t": 0,
                "miti": 0,
                "total_cost": 0
            })
        
        cost_col = f"cost_{method}"
        miti_col = f"Miti_{method}"
        
        # Prevent missing column errors (important)
        if cost_col not in row.index or miti_col not in row.index:
            return pd.Series({
                "cost_per_t": None,
                "miti": None,
                "total_cost": None
            })
        
        cost = row[cost_col]
        miti = row[miti_col]
        
        return pd.Series({
            "cost_per_t": cost,
            "miti": miti,
            "total_cost": cost * miti
        })
    
    values_tmp = assign_tmp.apply(extract_values, axis=1)
    
    # ======== Step 4: concatenate =========
    final_tmp = pd.concat([assign_tmp, values_tmp], axis=1)
    
    final_tmp = final_tmp[
        [
            "Plant_ID_High",
            "Plant_ID",
            "method",
            "cost_per_t",
            "miti",
            "total_cost"
        ]
    ]
    
    # ======== Step 5: save =========
    out_file = f"{name}_unit_level.xlsx"
    final_tmp.to_excel(os.path.join(base_path, out_file), index=False)
    
    # ======== Step 6: quick check =========
    print("  total miti:", final_tmp["miti"].sum())
    print("  total cost:", final_tmp["total_cost"].sum())
    print("  done\n")

Processing: 20pct
  total miti: 9353401303.4612
  total cost: 370095600546.4426
  done

Processing: 40pct
  total miti: 18706802831.846184
  total cost: 1277188993372.6863
  done

Processing: 60pct
  total miti: 28060204111.61211
  total cost: 2383565218978.407
  done

Processing: least_cost
  total miti: 9062074613.43701
  total cost: 781390862967.0872
  done

Processing: most_miti
  total miti: 32558722618.069218
  total cost: 3479724772725.032
  done

